# Lab Sheet 3: Developing a Tiny Transformer Language Model (complete version)

**Practical:** Build, train, evaluate, and save a Transformer from scratch

**Tools:** Google Colab, Python, PyTorch

**Dataset:** TinyStories

**Model:** Tiny character-level Transformer

This builds on the lecturer-provided `resources/tiny_transformer_llm.ipynb`. That notebook is missing
two things the lab sheet asks for: a `generate()` function (Step 9 — its own Gradio chat cell calls
`generate()` but never defines it) and a Step 10 experiment with model settings. Both are added below,
in the right place, so the notebook runs start to finish.

This notebook has not been run — run it top to bottom in Google Colab with a GPU runtime.

## Step 1 — Install and import libraries

In [ ]:
!pip -q install datasets

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Step 2 — Load the TinyStories dataset

In [ ]:
dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train[:1000]"
)

dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_text = "\n".join(dataset["train"]["text"])
val_text = "\n".join(dataset["test"]["text"])

print("Training stories:", len(dataset["train"]))
print("Validation stories:", len(dataset["test"]))
print(train_text[:500])

## Step 3 — Convert text into numerical IDs

Character-level tokenization: each unique character receives an integer ID.

In [ ]:
chars = sorted(set(train_text + val_text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(text):
    return [stoi[ch] for ch in text]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Encoded sample:", encode("Once upon a time")[:10])
print("Decoded sample:", decode(encode("Once upon a time")))

## Step 4 — Define the tiny Transformer

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, num_layers=2, nhead=2, dim_feedforward=128):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, 64)
        self.pos_emb = nn.Embedding(64, 64)

        layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,
            batch_first=True,
            activation="gelu"
        )

        self.blocks = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(64, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(positions)

        # Prevent each position from seeing future characters
        mask = torch.triu(
            torch.ones(T, T, device=idx.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.blocks(x, mask=mask)
        return self.lm_head(x)


model = TinyTransformer().to(device)

print("Number of parameters:", sum(p.numel() for p in model.parameters()))

## Step 5 — Prepare input and target sequences

In [ ]:
block_size = 64
batch_size = 32

def get_batch(split_name):
    source = train_data if split_name == "train" else val_data

    starts = torch.randint(len(source) - block_size - 1, (batch_size,))

    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])

    return x.to(device), y.to(device)


x, y = get_batch("train")

print("Input shape:", x.shape)
print("Target shape:", y.shape)

## Step 6 — Train the Transformer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

train_history = []
val_history = []

@torch.no_grad()
def evaluate(split_name, num_batches=10):
    model.eval()
    losses = []

    for _ in range(num_batches):
        x, y = get_batch(split_name)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)


for epoch in range(15):
    model.train()
    total_loss = 0.0

    for step in range(100):
        x, y = get_batch("train")
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / 100
    val_loss = evaluate("val", num_batches=10)

    train_history.append(train_loss)
    val_history.append(val_loss)

    print(f"Epoch {epoch + 1:02d} | Train loss: {train_loss:.3f} | Validation loss: {val_loss:.3f}")

## Step 7 — Plot the training and validation losses

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_history, label="Training loss")
plt.plot(val_history, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny Transformer Learning Curve")
plt.legend()
plt.show()

## Step 8 — Calculate validation perplexity

In [ ]:
val_loss = evaluate("val", num_batches=20)
val_perplexity = math.exp(min(val_loss, 20))

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {val_perplexity:.2f}")

## Step 9 — Generate stories

**This is the part missing from the lecturer's notebook** — its Gradio chat cell calls a `generate()`
function that is never defined. Defined here: feed in a starting prompt, and repeatedly predict one new
character at a time, sampling from the model's predicted probability distribution (the same
autoregressive, one-token-at-a-time process from the Week 3 slides).

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_chars=200):
    model.eval()

    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_chars):
        x = ids[:, -block_size:]
        logits = model(x)

        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)

    return decode(ids[0].tolist())


print(generate("Once upon a time", max_new_chars=200))

## Step 10 — Experiment with model settings

**Also missing from the lecturer's notebook.** `TinyTransformer` above now takes `num_layers`, `nhead`,
and `dim_feedforward` as arguments (the original hard-coded them), so different settings can be compared.
To actually compare them: create a variant below, substitute it for `model` in Step 6, retrain, and
compare its loss curve and generated text against the original 2-layer, 2-head model.

In [ ]:
# Example: a deeper model (4 layers instead of 2)
deeper_model = TinyTransformer(num_layers=4, nhead=2).to(device)
print("Deeper model parameters:", sum(p.numel() for p in deeper_model.parameters()))

# Example: more attention heads
wider_heads_model = TinyTransformer(num_layers=2, nhead=4).to(device)
print("More-heads model parameters:", sum(p.numel() for p in wider_heads_model.parameters()))

# More layers/heads generally lowers training loss faster, but on a dataset this small
# it also overfits faster — watch the validation loss in Step 6/7 when you try this.

## Step 11 — Save the trained model

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "chars": chars,
    "vocab_size": vocab_size,
    "config": {
        "embedding_dim": 64,
        "num_heads": 2,
        "num_layers": 2,
        "block_size": 64,
        "feedforward_dim": 128
    }
}

torch.save(checkpoint, "tiny_transformer.pt")
print("Model saved successfully!")

## Optional — Build a chat window with Gradio

Now works correctly, since `generate()` is defined above.

In [ ]:
!pip -q install gradio

import gradio as gr

def chat_with_transformer(message, history):
    if not message or not message.strip():
        return "Please enter a message."

    supported = "".join(ch for ch in message if ch in stoi)

    if not supported:
        return "Please try text using characters in the training dataset."

    try:
        return generate(supported, max_new_chars=100)
    except Exception as e:
        return f"Text generation failed: {type(e).__name__}: {e}"

demo = gr.ChatInterface(
    fn=chat_with_transformer,
    title="My Tiny Transformer Chatbot",
    description="A language model trained from scratch in our LLM class."
)

demo.launch()